<a href="https://colab.research.google.com/github/shivani11-glitch/seizure-detection/blob/main/week2_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — Training Loop


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## ConfidenceHeads Class

In [19]:
class ConfidenceHeads(nn.Module):
    def __init__(self):
        super(ConfidenceHeads, self).__init__()
        self.eeg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.ecg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.semg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.fusion_weights = nn.Parameter(torch.tensor([0.6, 0.2, 0.2]))

    def forward(self, x):
        pooled = torch.mean(x, dim=1)
        eeg_conf = self.eeg_head(pooled)
        ecg_conf = self.ecg_head(pooled)
        semg_conf = self.semg_head(pooled)
        weights = F.softmax(self.fusion_weights, dim=0)
        fused_conf = weights[0] * eeg_conf + weights[1] * ecg_conf + weights[2] * semg_conf  # (batch, 1)
        return eeg_conf, ecg_conf, semg_conf, fused_conf

SEIZURE_THRESHOLD = 0.5  #this
SUDEP_THRESHOLD = 0.75   #this

## Fake Data Function

In [20]:
def get_fake_batch(batch_size=16):
    x = torch.randn(batch_size, 32, 128)   #change this with p4 transformer output
    y = torch.randint(0, 2, (batch_size, 1)).float()    #real 0/1 labels from p1 dataset
    return x, y

## Model + Optimizer Setup

In [22]:
from google.colab import files
files.upload()
model = ConfidenceHeads()
model.load_state_dict(torch.load("confidence_heads.pt"))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.BCELoss()

Saving confidence_heads.pt to confidence_heads (2).pt


##Training Loop

In [23]:
losses = []
for epoch in range(30):
    x, y = get_fake_batch()
    eeg_conf, ecg_conf, semg_conf, fused_conf = model(x)
    loss_eeg = criterion(eeg_conf, y)
    loss_ecg = criterion(ecg_conf, y)
    loss_semg = criterion(semg_conf, y)
    loss_fused = criterion(fused_conf, y)
    total_loss = loss_eeg + loss_ecg + loss_semg + loss_fused
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(total_loss.item())

## Plot Loss Curve

In [25]:
# plt.plot(losses)
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss')
# plt.show()